In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Настройки оформления графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

CONST_NAMES = ('const', 'Intercept')


def fmt(x, decimals=3):
    """Число с заданным числом знаков; оценка, которая округляется до нуля, — с тремя значащими цифрами."""
    if pd.isna(x):
        return ''
    text = f'{x:.{decimals}f}'
    if float(text) == 0:
        return f'{x:.3g}' if x != 0 else f'{0:.{decimals}f}'
    return text


def fmt_p(p, decimals=3):
    """P-значение: меньше 0.001 печатается как «<0.001»."""
    return '<0.001' if p < 0.001 else f'{p:.{decimals}f}'


def errors_text(res, case='nom'):
    """Название типа стандартных ошибок подгонки: 'nom' — «неробастные», 'dat' — «по неробастным …»."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    if cov_type == 'nonrobust':
        return {'nom': 'неробастные стандартные ошибки',
                'dat': 'по неробастным стандартным ошибкам'}[case]
    return {'nom': f'робастные стандартные ошибки, {cov_type}',
            'dat': f'по робастным стандартным ошибкам {cov_type}'}[case]


def fit_ols(formula, data, robust=False, **kwargs):
    """МНК-подгонка; robust=True — робастные ошибки HC3. Всегда t- и F-распределения (use_t=True)."""
    cov_type = 'HC3' if robust else 'nonrobust'
    return ols(formula, data=data).fit(cov_type=cov_type, use_t=True, **kwargs)


def show_table(df, index=True, align=None):
    """Таблица pandas в Markdown; числа должны быть заранее превращены в строки через fmt."""
    display(Markdown('\n\n' + df.to_markdown(index=index, colalign=align, disable_numparse=True) + '\n\n'))


def nobs_text(res):
    """Строка «Модель была подогнана по N наблюдениям»."""
    n = int(res.nobs)
    word = 'наблюдению' if n % 10 == 1 and n % 100 != 11 else 'наблюдениям'
    return f'Модель была подогнана по {n} {word}.'


def show_coefs(res, decimals=3, show_nobs=True):
    """Оценки коэффициентов: вертикальная таблица «Переменная | Оценка» и число наблюдений."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    column = 'Оценка' if cov_type == 'nonrobust' else f'Оценка ({cov_type})'
    table = pd.DataFrame({'Переменная': res.params.index,
                          column: [fmt(b, decimals) for b in res.params]})
    text = table.to_markdown(index=False, colalign=('left', 'right'), disable_numparse=True)
    if show_nobs:
        text += '\n\n' + nobs_text(res)
    display(Markdown('\n\n' + text + '\n\n'))


def display_answer(text, use_callout=True, theme='note', title='Критическое значение'):
    """Ответ в рамке (callout): theme='note' — число, 'tip' — вывод, 'important' — условие."""
    if use_callout:
        head = f'title="{title}" ' if title else ''
        output_md = f'\n\n::: {{.callout-{theme} {head}icon="true"}}\n{text}\n:::\n\n'
    elif title:
        output_md = f'\n\n**{title}:** {text}\n\n'
    else:
        output_md = f'\n\n{text}\n\n'
    display(Markdown(output_md))


def show_conditions(level, errors=None, hypothesis=None, extra=()):
    """Блок «Условия теста»: уровень значимости (доля или список долей), тип ошибок, гипотеза формулой."""
    levels = level if isinstance(level, (list, tuple)) else [level]
    items = ['уровень значимости: **' + ' и '.join(f'{a * 100:g}%' for a in levels) + '**']
    if errors:
        items.append(f'стандартные ошибки: **{errors}**')
    hypotheses = [hypothesis] if isinstance(hypothesis, str) else list(hypothesis or [])
    if len(hypotheses) == 1:
        items.append(f'гипотеза: {hypotheses[0]}')
    else:
        items += [f'гипотеза {i}: {h}' for i, h in enumerate(hypotheses, start=1)]
    items += list(extra)
    display_answer('\n'.join(f'- {item}' for item in items), theme='important', title='Условия теста')


def signif_stars(p):
    """Коды значимости: *** p<0.01, ** p<0.05, * p<0.1."""
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''


SIGNIF_LEGEND = '*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$'


def print_regression_summary(res, decimals=3, show_signif=True):
    """Таблица t-теста: Coef., Std.Err., t-value, p-value, Signif. Заголовок называет тип ошибок."""
    title = f'**Результаты $t$-теста для коэффициентов ({errors_text(res)}):**'
    table = pd.DataFrame({
        'Coef.': [fmt(b, decimals) for b in res.params],
        'Std.Err.': [fmt(s, decimals) for s in res.bse],
        't-value': [fmt(t, decimals) for t in res.tvalues],
        'p-value': [fmt_p(p, decimals) for p in res.pvalues],
    }, index=res.params.index)
    legend = ''
    if show_signif:
        table['Signif.'] = ['' if name in CONST_NAMES else signif_stars(p)
                            for name, p in res.pvalues.items()]
        legend = '\n\n' + SIGNIF_LEGEND
    align = ('left',) + ('right',) * 4 + (('left',) if show_signif else ())
    display(Markdown(f'\n\n{title}\n\n' + table.to_markdown(colalign=align, disable_numparse=True) + legend + '\n\n'))


def print_significant_coeffs(res, sign_level=0.05, theme='note', title=None):
    """Рамка со списком коэффициентов, значимых на уровне sign_level (без константы)."""
    names = [name for name, p in res.pvalues.items() if p < sign_level and name not in CONST_NAMES]
    if title is None:
        title = f'Коэффициенты, значимые на уровне {sign_level * 100:g}% ({errors_text(res, "dat")})'
    text = ', '.join(f'`{name}`' for name in names) if names else 'Значимых коэффициентов не выявлено.'
    display_answer(text, theme=theme, title=title)


def show_models(models, names=None, decimals=3, regressor_order=(), errors_note=None, info=None):
    """Сравнение регрессий (summary_col): строка Dep. Variable, коды значимости, пояснение про скобки.

    info — словарь {название строки: функция от подгонки} для дополнительных строк (N, F и т. п.).
    """
    table = summary_col(models, model_names=names, stars=True, float_format=f'%0.{decimals}f',
                        regressor_order=list(regressor_order), info_dict=info or {}).tables[0]
    top = pd.DataFrame({col: [m.model.endog_names] for col, m in zip(table.columns, models)},
                       index=['Dep. Variable'])
    table = pd.concat([top, table])
    if errors_note is None:
        errors_note = errors_text(models[0])
    note = f'{SIGNIF_LEGEND}. В скобках — {errors_note}.'
    display(Markdown('\n\n' + table.to_markdown(disable_numparse=True) + '\n\n' + note + '\n\n'))


def show_ci(res, level=0.95, decimals=3):
    """Доверительные интервалы: «Переменная | Нижняя граница | Верхняя граница», заголовок над таблицей."""
    ci = res.conf_int(alpha=1 - level)
    table = pd.DataFrame({'Переменная': ci.index,
                          'Нижняя граница': [fmt(x, decimals) for x in ci[0]],
                          'Верхняя граница': [fmt(x, decimals) for x in ci[1]]})
    title = f'**{level * 100:g}%-е доверительные интервалы ({errors_text(res)}):**'
    display(Markdown(f'\n\n{title}\n\n'
                     + table.to_markdown(index=False, colalign=('left', 'right', 'right'), disable_numparse=True) + '\n\n'))


# Загрузка данных
sleep75 = pd.read_csv('../datasets/sleep75.csv')
Labour = pd.read_csv('../datasets/Labour.csv')
Electricity = pd.read_csv('../datasets/Electricity.csv')

# Помощники листка

H0_ZERO = r'$H_0:\beta_j=0$ для каждого коэффициента'


def show_crit(res, sign_level, decimals=3):
    """Двустороннее критическое значение распределения Стьюдента: квантиль, степени свободы, число."""
    t_cr = stats.t.ppf(1 - sign_level / 2, res.df_resid)
    display_answer(f'$t_{{crit}} = t_{{{1 - sign_level / 2:g}}}({int(res.df_resid)}) = {t_cr:.{decimals}f}$')


def signif_answer(formula, data, sign_level, robust=False):
    """Ответ раздела 1: таблица t-теста, число наблюдений, критическое значение, значимые коэффициенты."""
    res = fit_ols(formula, data, robust=robust)
    print_regression_summary(res)
    display(Markdown(nobs_text(res)))
    show_crit(res, sign_level)
    print_significant_coeffs(res, sign_level)


def show_regr(formula, data):
    """Условие раздела 2: оценки с неробастными (1) и HC3 (2) ошибками, четыре значащие цифры."""
    res, res_hc = fit_ols(formula, data), fit_ols(formula, data, robust=True)
    order = [name for name in res.params.index if name not in CONST_NAMES] + ['Intercept']
    table = summary_col(
        [res, res_hc], model_names=['(1)', '(2)'], stars=True, float_format='%.4g', regressor_order=order,
        info_dict={'Residual Std. Error': lambda m: f'{np.sqrt(m.mse_resid):.4g}',
                   'F Statistic': lambda m: f'{float(np.squeeze(m.fvalue)):.4g}'}).tables[0]
    top = pd.DataFrame({col: [res.model.endog_names] for col in table.columns}, index=['Dep. Variable'])
    note = f'{SIGNIF_LEGEND}. В скобках — стандартные ошибки: (1) неробастные, (2) HC3.'
    display(Markdown('\n\n' + pd.concat([top, table]).to_markdown(disable_numparse=True)
                     + f'\n\n{note}\n\n{nobs_text(res)}\n\n'))
    return res, res_hc


def theta_test(res, name, theta, sign_level, decimals=2):
    """t-тест гипотезы H0: beta = theta по ошибкам подгонки res: оценка, ошибка, статистика и вывод."""
    b, se = res.params[name], res.bse[name]
    t_stat, t_cr = (b - theta) / se, stats.t.ppf(1 - sign_level / 2, res.df_resid)
    b_text, se_text = f'{b:.4g}', f'{se:.4g}'
    # расчёт по показанным четырём значащим цифрам обязан давать тот же ответ, что и точный
    assert fmt((float(b_text) - theta) / float(se_text), decimals) == fmt(t_stat, decimals), name
    se_column = 'Стандартная ошибка' + ('' if res.cov_type == 'nonrobust' else f' ({res.cov_type})')
    show_table(pd.DataFrame({'Переменная': [name], 'Оценка': [b_text], se_column: [se_text],
                             'Тестовая статистика': [fmt(t_stat, decimals)]}),
               index=False, align=('left', 'right', 'right', 'right'))
    sign, verdict = ('>', 'отвергается') if abs(t_stat) > t_cr else ('<', 'не отвергается')
    display_answer(f'$\\lvert t\\rvert = {abs(t_stat):.{decimals}f} {sign} {t_cr:.{decimals}f}$, '
                   f'гипотеза $H_0$ {verdict}.', theme='tip', title='Вывод')

::: {.callout-important title="Важно"}
- Логарифм натуральный; константа включается в модель по умолчанию.
- Данные лежат в папке `datasets`, описание переменных — в `datasets/data-description.md`.
- Округление указано в каждой задаче: 3 знака в первом разделе, 2 знака во втором. Тестовая статистика считается по неокруглённым оценкам.
- Коэффициенты подписаны так, как их печатает statsmodels: `Intercept` — константа, `np.log(x)` — $\log x$, `I(np.log(x) ** 2)` — $\log^2 x$.
- Индекс коэффициента в гипотезе — имя переменной: $\beta_{capital}$ — коэффициент при `np.log(capital)`. Альтернатива двусторонняя.
- Неробастные стандартные ошибки — обычные ошибки МНК, робастные — HC3; критические значения и P-значения — по t-распределению (`fit(cov_type='HC3', use_t=True)`).
:::

# t-тест: значимость коэффициентов

## sleep equation #1

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + south + male + smsa + marr + union`

Переменные `sleep` и `totwrk` измерены в минутах в неделю.

In [2]:
#| output: asis
show_conditions(0.05, 'неробастные', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **неробастные**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [3]:
#| output: asis
signif_answer('sleep ~ totwrk + age + south + male + smsa + marr + union', sleep75, 0.05)



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|           |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:----------|---------:|-----------:|----------:|----------:|:----------|
| Intercept | 3441.297 |     78.942 |    43.593 |    <0.001 |           |
| totwrk    |   -0.169 |      0.018 |    -9.363 |    <0.001 | ***       |
| age       |    2.841 |      1.387 |     2.048 |     0.041 | **        |
| south     |  101.676 |     41.834 |     2.430 |     0.015 | **        |
| male      |   86.038 |     34.906 |     2.465 |     0.014 | **        |
| smsa      |  -53.925 |     33.156 |    -1.626 |     0.104 |           |
| marr      |   30.006 |     41.910 |     0.716 |     0.474 |           |
| union     |   12.440 |     38.096 |     0.327 |     0.744 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 706 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.975}(698) = 1.963$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по неробастным стандартным ошибкам)" icon="true"}
`totwrk`, `age`, `south`, `male`
:::



## sleep equation #2

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + south + male + smsa + marr + union`

In [4]:
#| output: asis
show_conditions(0.05, 'робастные (HC3)', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **робастные (HC3)**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [5]:
#| output: asis
signif_answer('sleep ~ totwrk + age + south + male + smsa + marr + union', sleep75, 0.05, robust=True)



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|           |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:----------|---------:|-----------:|----------:|----------:|:----------|
| Intercept | 3441.297 |     88.891 |    38.714 |    <0.001 |           |
| totwrk    |   -0.169 |      0.020 |    -8.299 |    <0.001 | ***       |
| age       |    2.841 |      1.362 |     2.086 |     0.037 | **        |
| south     |  101.676 |     42.315 |     2.403 |     0.017 | **        |
| male      |   86.038 |     35.758 |     2.406 |     0.016 | **        |
| smsa      |  -53.925 |     33.372 |    -1.616 |     0.107 |           |
| marr      |   30.006 |     46.090 |     0.651 |     0.515 |           |
| union     |   12.440 |     37.277 |     0.334 |     0.739 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 706 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.975}(698) = 1.963$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по робастным стандартным ошибкам HC3)" icon="true"}
`totwrk`, `age`, `south`, `male`
:::



## sleep equation #3

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + south + male + smsa + marr + union + np.log(hrwage)`

Переменная `hrwage` известна не для всех наблюдений: модель подгоняется по 532 наблюдениям из 706.

In [6]:
#| output: asis
show_conditions(0.10, 'неробастные', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **10%**
- стандартные ошибки: **неробастные**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [7]:
#| output: asis
signif_answer('sleep ~ totwrk + age + south + male + smsa + marr + union + np.log(hrwage)', sleep75, 0.10)



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|                |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|---------:|-----------:|----------:|----------:|:----------|
| Intercept      | 3453.355 |     93.371 |    36.985 |    <0.001 |           |
| totwrk         |   -0.159 |      0.021 |    -7.629 |    <0.001 | ***       |
| age            |    1.950 |      1.584 |     1.230 |     0.219 |           |
| south          |   80.244 |     45.216 |     1.775 |     0.077 | *         |
| male           |   41.098 |     43.049 |     0.955 |     0.340 |           |
| smsa           |  -36.093 |     38.206 |    -0.945 |     0.345 |           |
| marr           |   58.917 |     46.613 |     1.264 |     0.207 |           |
| union          |   24.479 |     42.948 |     0.570 |     0.569 |           |
| np.log(hrwage) |   -2.590 |     32.197 |    -0.080 |     0.936 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 532 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.95}(523) = 1.648$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 10% (по неробастным стандартным ошибкам)" icon="true"}
`totwrk`, `south`
:::



## sleep equation #4

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + south + male + smsa + marr + union + np.log(hrwage)`

Переменная `hrwage` известна не для всех наблюдений: модель подгоняется по 532 наблюдениям из 706.

In [8]:
#| output: asis
show_conditions(0.10, 'робастные (HC3)', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **10%**
- стандартные ошибки: **робастные (HC3)**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [9]:
#| output: asis
signif_answer('sleep ~ totwrk + age + south + male + smsa + marr + union + np.log(hrwage)', sleep75, 0.10,
              robust=True)



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|                |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:---------------|---------:|-----------:|----------:|----------:|:----------|
| Intercept      | 3453.355 |     98.213 |    35.162 |    <0.001 |           |
| totwrk         |   -0.159 |      0.025 |    -6.354 |    <0.001 | ***       |
| age            |    1.950 |      1.571 |     1.241 |     0.215 |           |
| south          |   80.244 |     47.401 |     1.693 |     0.091 | *         |
| male           |   41.098 |     42.778 |     0.961 |     0.337 |           |
| smsa           |  -36.093 |     38.580 |    -0.936 |     0.350 |           |
| marr           |   58.917 |     48.350 |     1.219 |     0.224 |           |
| union          |   24.479 |     43.006 |     0.569 |     0.569 |           |
| np.log(hrwage) |   -2.590 |     32.992 |    -0.079 |     0.937 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 532 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.95}(523) = 1.648$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 10% (по робастным стандартным ошибкам HC3)" icon="true"}
`totwrk`, `south`
:::



## output equation #1

Для набора данных `Labour` рассмотрим регрессию

```python
np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage) +
    I(np.log(capital)**2) + I(np.log(labour)**2)
```

Переменные `output` и `capital` измерены в млн евро, `labour` — число работников, `wage` — в тыс. евро на работника.

In [10]:
#| output: asis
show_conditions(0.10, 'неробастные', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **10%**
- стандартные ошибки: **неробастные**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [11]:
#| output: asis
signif_answer('np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage)'
              ' + I(np.log(capital)**2) + I(np.log(labour)**2)', Labour, 0.10)



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|                         |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept               |  -4.258 |      0.257 |   -16.599 |    <0.001 |           |
| np.log(capital)         |   0.140 |      0.014 |     9.743 |    <0.001 | ***       |
| np.log(labour)          |   0.473 |      0.071 |     6.672 |    <0.001 | ***       |
| np.log(wage)            |   0.853 |      0.058 |    14.751 |    <0.001 | ***       |
| I(np.log(capital) ** 2) |   0.008 |      0.004 |     1.735 |     0.083 | *         |
| I(np.log(labour) ** 2)  |   0.027 |      0.008 |     3.355 |    <0.001 | ***       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 569 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.95}(563) = 1.648$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 10% (по неробастным стандартным ошибкам)" icon="true"}
`np.log(capital)`, `np.log(labour)`, `np.log(wage)`, `I(np.log(capital) ** 2)`, `I(np.log(labour) ** 2)`
:::



## output equation #2

Для набора данных `Labour` рассмотрим регрессию

```python
np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage) +
    I(np.log(capital)**2) + I(np.log(labour)**2)
```

In [12]:
#| output: asis
show_conditions(0.10, 'робастные (HC3)', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **10%**
- стандартные ошибки: **робастные (HC3)**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [13]:
#| output: asis
signif_answer('np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage)'
              ' + I(np.log(capital)**2) + I(np.log(labour)**2)', Labour, 0.10, robust=True)



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|                         |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept               |  -4.258 |      0.547 |    -7.782 |    <0.001 |           |
| np.log(capital)         |   0.140 |      0.030 |     4.736 |    <0.001 | ***       |
| np.log(labour)          |   0.473 |      0.197 |     2.404 |     0.017 | **        |
| np.log(wage)            |   0.853 |      0.086 |     9.929 |    <0.001 | ***       |
| I(np.log(capital) ** 2) |   0.008 |      0.008 |     0.932 |     0.352 |           |
| I(np.log(labour) ** 2)  |   0.027 |      0.020 |     1.376 |     0.169 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 569 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.95}(563) = 1.648$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 10% (по робастным стандартным ошибкам HC3)" icon="true"}
`np.log(capital)`, `np.log(labour)`, `np.log(wage)`
:::



## cost equation #1

Для набора данных `Electricity` рассмотрим регрессию

```python
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
    np.log(pl) + np.log(pk) + np.log(pf)
```

Переменные: `cost` — общие издержки, `q` — выпуск, `pl`, `pk`, `pf` — цены труда, капитала и топлива.

In [14]:
#| output: asis
show_conditions(0.01, 'неробастные', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [15]:
#| output: asis
signif_answer('np.log(cost) ~ np.log(q) + I(np.log(q)**2) + np.log(pl) + np.log(pk) + np.log(pf)',
              Electricity, 0.01)



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|                   |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept         |  -6.739 |      0.706 |    -9.541 |    <0.001 |           |
| np.log(q)         |   0.403 |      0.032 |    12.734 |    <0.001 | ***       |
| I(np.log(q) ** 2) |   0.030 |      0.002 |    14.024 |    <0.001 | ***       |
| np.log(pl)        |   0.146 |      0.070 |     2.073 |     0.040 | **        |
| np.log(pk)        |   0.157 |      0.058 |     2.721 |     0.007 | ***       |
| np.log(pf)        |   0.685 |      0.043 |    16.043 |    <0.001 | ***       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 158 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.995}(152) = 2.609$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по неробастным стандартным ошибкам)" icon="true"}
`np.log(q)`, `I(np.log(q) ** 2)`, `np.log(pk)`, `np.log(pf)`
:::



## cost equation #2

Для набора данных `Electricity` рассмотрим регрессию

```python
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
    np.log(pl) + np.log(pk) + np.log(pf)
```

In [16]:
#| output: asis
show_conditions(0.01, 'робастные (HC3)', H0_ZERO)



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **робастные (HC3)**
- гипотеза: $H_0:\beta_j=0$ для каждого коэффициента
:::



- Оцените регрессию.
- Выведите результаты t-теста.
- Вычислите критическое значение.
- Какие коэффициенты значимы?

**Ответ округлите до 3-х десятичных знаков.**

In [17]:
#| output: asis
signif_answer('np.log(cost) ~ np.log(q) + I(np.log(q)**2) + np.log(pl) + np.log(pk) + np.log(pf)',
              Electricity, 0.01, robust=True)



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|                   |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept         |  -6.739 |      0.847 |    -7.954 |    <0.001 |           |
| np.log(q)         |   0.403 |      0.066 |     6.086 |    <0.001 | ***       |
| I(np.log(q) ** 2) |   0.030 |      0.004 |     7.419 |    <0.001 | ***       |
| np.log(pl)        |   0.146 |      0.085 |     1.711 |     0.089 | *         |
| np.log(pk)        |   0.157 |      0.062 |     2.522 |     0.013 | **        |
| np.log(pf)        |   0.685 |      0.052 |    13.188 |    <0.001 | ***       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 158 наблюдениям.



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.995}(152) = 2.609$
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 1% (по робастным стандартным ошибкам HC3)" icon="true"}
`np.log(q)`, `I(np.log(q) ** 2)`, `np.log(pf)`
:::



# t-тест: равенство коэффициента заданному числу

## output equation #3

Для набора данных `Labour` рассмотрим регрессию **np.log(output) на np.log(capital), np.log(labour)**.

Результаты оценивания:

In [18]:
#| output: asis
res, res_hc = show_regr('np.log(output) ~ np.log(capital) + np.log(labour)', Labour)



|                     | (1)            | (2)            |
|:--------------------|:---------------|:---------------|
| Dep. Variable       | np.log(output) | np.log(output) |
| np.log(capital)     | 0.2076***      | 0.2076***      |
|                     | (0.01719)      | (0.03159)      |
| np.log(labour)      | 0.7148***      | 0.7148***      |
|                     | (0.02314)      | (0.04362)      |
| Intercept           | -1.711***      | -1.711***      |
|                     | (0.09671)      | (0.1841)       |
| R-squared           | 0.8378         | 0.8378         |
| R-squared Adj.      | 0.8373         | 0.8373         |
| F Statistic         | 1462           | 486.1          |
| Residual Std. Error | 0.4781         | 0.4781         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — стандартные ошибки: (1) неробастные, (2) HC3.

Модель была подогнана по 569 наблюдениям.



In [19]:
#| output: asis
show_conditions(0.01, 'неробастные и робастные (HC3)', r'$H_0:\beta_{capital}=0.5$')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3)**
- гипотеза: $H_0:\beta_{capital}=0.5$
:::



Вычислите тестовую статистику. Вычислите критическое значение. Какой можно сделать вывод?

**Ответ округлите до 2-х десятичных знаков.**

In [20]:
#| output: asis
show_crit(res, 0.01, decimals=2)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.995}(566) = 2.58$
:::



**Неробастный t-тест**

In [21]:
#| output: asis
theta_test(res, 'np.log(capital)', 0.5, 0.01)



| Переменная      |   Оценка |   Стандартная ошибка |   Тестовая статистика |
|:----------------|---------:|---------------------:|----------------------:|
| np.log(capital) |   0.2076 |              0.01719 |                -17.01 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 17.01 > 2.58$, гипотеза $H_0$ отвергается.
:::



**Робастный t-тест (HC3)**

In [22]:
#| output: asis
theta_test(res_hc, 'np.log(capital)', 0.5, 0.01)



| Переменная      |   Оценка |   Стандартная ошибка (HC3) |   Тестовая статистика |
|:----------------|---------:|---------------------------:|----------------------:|
| np.log(capital) |   0.2076 |                    0.03159 |                 -9.26 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 9.26 > 2.58$, гипотеза $H_0$ отвергается.
:::



## output equation #4

Для набора данных `Labour` рассмотрим регрессию **np.log(output) на np.log(capital), np.log(labour)**.

Результаты оценивания:

In [23]:
#| output: asis
res, res_hc = show_regr('np.log(output) ~ np.log(capital) + np.log(labour)', Labour)



|                     | (1)            | (2)            |
|:--------------------|:---------------|:---------------|
| Dep. Variable       | np.log(output) | np.log(output) |
| np.log(capital)     | 0.2076***      | 0.2076***      |
|                     | (0.01719)      | (0.03159)      |
| np.log(labour)      | 0.7148***      | 0.7148***      |
|                     | (0.02314)      | (0.04362)      |
| Intercept           | -1.711***      | -1.711***      |
|                     | (0.09671)      | (0.1841)       |
| R-squared           | 0.8378         | 0.8378         |
| R-squared Adj.      | 0.8373         | 0.8373         |
| F Statistic         | 1462           | 486.1          |
| Residual Std. Error | 0.4781         | 0.4781         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — стандартные ошибки: (1) неробастные, (2) HC3.

Модель была подогнана по 569 наблюдениям.



In [24]:
#| output: asis
show_conditions(0.10, 'неробастные и робастные (HC3)', r'$H_0:\beta_{labour}=0.7$')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **10%**
- стандартные ошибки: **неробастные и робастные (HC3)**
- гипотеза: $H_0:\beta_{labour}=0.7$
:::



Вычислите тестовую статистику. Вычислите критическое значение. Какой можно сделать вывод?

**Ответ округлите до 2-х десятичных знаков.**

In [25]:
#| output: asis
show_crit(res, 0.10, decimals=2)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.95}(566) = 1.65$
:::



**Неробастный t-тест**

In [26]:
#| output: asis
theta_test(res, 'np.log(labour)', 0.7, 0.10)



| Переменная     |   Оценка |   Стандартная ошибка |   Тестовая статистика |
|:---------------|---------:|---------------------:|----------------------:|
| np.log(labour) |   0.7148 |              0.02314 |                  0.64 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 0.64 < 1.65$, гипотеза $H_0$ не отвергается.
:::



**Робастный t-тест (HC3)**

In [27]:
#| output: asis
theta_test(res_hc, 'np.log(labour)', 0.7, 0.10)



| Переменная     |   Оценка |   Стандартная ошибка (HC3) |   Тестовая статистика |
|:---------------|---------:|---------------------------:|----------------------:|
| np.log(labour) |   0.7148 |                    0.04362 |                  0.34 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 0.34 < 1.65$, гипотеза $H_0$ не отвергается.
:::



## cost equation #3

Для набора данных `Electricity` рассмотрим регрессию **np.log(cost) на np.log(q), np.log(pl), np.log(pk), np.log(pf)**.

Результаты оценивания:

In [28]:
#| output: asis
res, res_hc = show_regr('np.log(cost) ~ np.log(q) + np.log(pl) + np.log(pk) + np.log(pf)', Electricity)



|                     | (1)          | (2)          |
|:--------------------|:-------------|:-------------|
| Dep. Variable       | np.log(cost) | np.log(cost) |
| np.log(q)           | 0.8381***    | 0.8381***    |
|                     | (0.009387)   | (0.0205)     |
| np.log(pl)          | 0.04426      | 0.04426      |
|                     | (0.1058)     | (0.119)      |
| np.log(pk)          | 0.1884**     | 0.1884**     |
|                     | (0.08707)    | (0.089)      |
| np.log(pf)          | 0.7131***    | 0.7131***    |
|                     | (0.06436)    | (0.0814)     |
| Intercept           | -7.472***    | -7.472***    |
|                     | (1.063)      | (1.156)      |
| R-squared           | 0.9822       | 0.9822       |
| R-squared Adj.      | 0.9817       | 0.9817       |
| F Statistic         | 2107         | 552.7        |
| Residual Std. Error | 0.2087       | 0.2087       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — стандартные ошибки: (1) неробастные, (2) HC3.

Модель была подогнана по 158 наблюдениям.



In [29]:
#| output: asis
show_conditions(0.05, 'неробастные и робастные (HC3)', r'$H_0:\beta_{q}=1$')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **неробастные и робастные (HC3)**
- гипотеза: $H_0:\beta_{q}=1$
:::



Вычислите тестовую статистику. Вычислите критическое значение. Какой можно сделать вывод?

**Ответ округлите до 2-х десятичных знаков.**

In [30]:
#| output: asis
show_crit(res, 0.05, decimals=2)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.975}(153) = 1.98$
:::



**Неробастный t-тест**

In [31]:
#| output: asis
theta_test(res, 'np.log(q)', 1, 0.05)



| Переменная   |   Оценка |   Стандартная ошибка |   Тестовая статистика |
|:-------------|---------:|---------------------:|----------------------:|
| np.log(q)    |   0.8381 |             0.009387 |                -17.25 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 17.25 > 1.98$, гипотеза $H_0$ отвергается.
:::



**Робастный t-тест (HC3)**

In [32]:
#| output: asis
theta_test(res_hc, 'np.log(q)', 1, 0.05)



| Переменная   |   Оценка |   Стандартная ошибка (HC3) |   Тестовая статистика |
|:-------------|---------:|---------------------------:|----------------------:|
| np.log(q)    |   0.8381 |                     0.0205 |                 -7.90 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 7.90 > 1.98$, гипотеза $H_0$ отвергается.
:::



## cost equation #4

Для набора данных `Electricity` рассмотрим регрессию **np.log(cost) на np.log(q), np.log(pl), np.log(pk), np.log(pf)**.

Результаты оценивания:

In [33]:
#| output: asis
res, res_hc = show_regr('np.log(cost) ~ np.log(q) + np.log(pl) + np.log(pk) + np.log(pf)', Electricity)



|                     | (1)          | (2)          |
|:--------------------|:-------------|:-------------|
| Dep. Variable       | np.log(cost) | np.log(cost) |
| np.log(q)           | 0.8381***    | 0.8381***    |
|                     | (0.009387)   | (0.0205)     |
| np.log(pl)          | 0.04426      | 0.04426      |
|                     | (0.1058)     | (0.119)      |
| np.log(pk)          | 0.1884**     | 0.1884**     |
|                     | (0.08707)    | (0.089)      |
| np.log(pf)          | 0.7131***    | 0.7131***    |
|                     | (0.06436)    | (0.0814)     |
| Intercept           | -7.472***    | -7.472***    |
|                     | (1.063)      | (1.156)      |
| R-squared           | 0.9822       | 0.9822       |
| R-squared Adj.      | 0.9817       | 0.9817       |
| F Statistic         | 2107         | 552.7        |
| Residual Std. Error | 0.2087       | 0.2087       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — стандартные ошибки: (1) неробастные, (2) HC3.

Модель была подогнана по 158 наблюдениям.



In [34]:
#| output: asis
show_conditions(0.10, 'неробастные и робастные (HC3)', r'$H_0:\beta_{pf}=1$')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **10%**
- стандартные ошибки: **неробастные и робастные (HC3)**
- гипотеза: $H_0:\beta_{pf}=1$
:::



Вычислите тестовую статистику. Вычислите критическое значение. Какой можно сделать вывод?

**Ответ округлите до 2-х десятичных знаков.**

In [35]:
#| output: asis
show_crit(res, 0.10, decimals=2)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.95}(153) = 1.65$
:::



**Неробастный t-тест**

In [36]:
#| output: asis
theta_test(res, 'np.log(pf)', 1, 0.10)



| Переменная   |   Оценка |   Стандартная ошибка |   Тестовая статистика |
|:-------------|---------:|---------------------:|----------------------:|
| np.log(pf)   |   0.7131 |              0.06436 |                 -4.46 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 4.46 > 1.65$, гипотеза $H_0$ отвергается.
:::



**Робастный t-тест (HC3)**

In [37]:
#| output: asis
theta_test(res_hc, 'np.log(pf)', 1, 0.10)



| Переменная   |   Оценка |   Стандартная ошибка (HC3) |   Тестовая статистика |
|:-------------|---------:|---------------------------:|----------------------:|
| np.log(pf)   |   0.7131 |                     0.0814 |                 -3.52 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 3.52 > 1.65$, гипотеза $H_0$ отвергается.
:::



## sleep equation #5

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + south + male + smsa + yngkid + marr + union`

Результаты оценивания:

In [38]:
#| output: asis
res, res_hc = show_regr('sleep ~ totwrk + age + south + male + smsa + yngkid + marr + union', sleep75)



|                     | (1)        | (2)        |
|:--------------------|:-----------|:-----------|
| Dep. Variable       | sleep      | sleep      |
| totwrk              | -0.1691*** | -0.1691*** |
|                     | (0.01807)  | (0.02035)  |
| age                 | 2.714*     | 2.714*     |
|                     | (1.472)    | (1.451)    |
| south               | 102.3**    | 102.3**    |
|                     | (41.93)    | (42.59)    |
| male                | 87.11**    | 87.11**    |
|                     | (35.17)    | (35.96)    |
| smsa                | -54.19     | -54.19     |
|                     | (33.19)    | (33.45)    |
| yngkid              | -13.05     | -13.05     |
|                     | (50.46)    | (54.58)    |
| marr                | 31.36      | 31.36      |
|                     | (42.26)    | (46.27)    |
| union               | 11.87      | 11.87      |
|                     | (38.19)    | (37.53)    |
| Intercept           | 3447***    | 3447***    |
|                     | (81.84)    | (91.67)    |
| R-squared           | 0.1314     | 0.1314     |
| R-squared Adj.      | 0.1214     | 0.1214     |
| F Statistic         | 13.18      | 10.44      |
| Residual Std. Error | 416.6      | 416.6      |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — стандартные ошибки: (1) неробастные, (2) HC3.

Модель была подогнана по 706 наблюдениям.



In [39]:
#| output: asis
show_conditions(0.01, 'неробастные и робастные (HC3)', r'$H_0:\beta_{male}=90$')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3)**
- гипотеза: $H_0:\beta_{male}=90$
:::



Вычислите тестовую статистику. Вычислите критическое значение. Какой можно сделать вывод?

**Ответ округлите до 2-х десятичных знаков.**

In [40]:
#| output: asis
show_crit(res, 0.01, decimals=2)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.995}(697) = 2.58$
:::



**Неробастный t-тест**

In [41]:
#| output: asis
theta_test(res, 'male', 90, 0.01)



| Переменная   |   Оценка |   Стандартная ошибка |   Тестовая статистика |
|:-------------|---------:|---------------------:|----------------------:|
| male         |    87.11 |                35.17 |                 -0.08 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 0.08 < 2.58$, гипотеза $H_0$ не отвергается.
:::



**Робастный t-тест (HC3)**

In [42]:
#| output: asis
theta_test(res_hc, 'male', 90, 0.01)



| Переменная   |   Оценка |   Стандартная ошибка (HC3) |   Тестовая статистика |
|:-------------|---------:|---------------------------:|----------------------:|
| male         |    87.11 |                      35.96 |                 -0.08 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 0.08 < 2.58$, гипотеза $H_0$ не отвергается.
:::



## sleep equation #6

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + south + male + smsa + yngkid + marr + union`

Результаты оценивания:

In [43]:
#| output: asis
res, res_hc = show_regr('sleep ~ totwrk + age + south + male + smsa + yngkid + marr + union', sleep75)



|                     | (1)        | (2)        |
|:--------------------|:-----------|:-----------|
| Dep. Variable       | sleep      | sleep      |
| totwrk              | -0.1691*** | -0.1691*** |
|                     | (0.01807)  | (0.02035)  |
| age                 | 2.714*     | 2.714*     |
|                     | (1.472)    | (1.451)    |
| south               | 102.3**    | 102.3**    |
|                     | (41.93)    | (42.59)    |
| male                | 87.11**    | 87.11**    |
|                     | (35.17)    | (35.96)    |
| smsa                | -54.19     | -54.19     |
|                     | (33.19)    | (33.45)    |
| yngkid              | -13.05     | -13.05     |
|                     | (50.46)    | (54.58)    |
| marr                | 31.36      | 31.36      |
|                     | (42.26)    | (46.27)    |
| union               | 11.87      | 11.87      |
|                     | (38.19)    | (37.53)    |
| Intercept           | 3447***    | 3447***    |
|                     | (81.84)    | (91.67)    |
| R-squared           | 0.1314     | 0.1314     |
| R-squared Adj.      | 0.1214     | 0.1214     |
| F Statistic         | 13.18      | 10.44      |
| Residual Std. Error | 416.6      | 416.6      |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$. В скобках — стандартные ошибки: (1) неробастные, (2) HC3.

Модель была подогнана по 706 наблюдениям.



In [44]:
#| output: asis
show_conditions(0.01, 'неробастные и робастные (HC3)', r'$H_0:\beta_{south}=100$')



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **1%**
- стандартные ошибки: **неробастные и робастные (HC3)**
- гипотеза: $H_0:\beta_{south}=100$
:::



Вычислите тестовую статистику. Вычислите критическое значение. Какой можно сделать вывод?

**Ответ округлите до 2-х десятичных знаков.**

In [45]:
#| output: asis
show_crit(res, 0.01, decimals=2)



::: {.callout-note title="Критическое значение" icon="true"}
$t_{crit} = t_{0.995}(697) = 2.58$
:::



**Неробастный t-тест**

In [46]:
#| output: asis
theta_test(res, 'south', 100, 0.01)



| Переменная   |   Оценка |   Стандартная ошибка |   Тестовая статистика |
|:-------------|---------:|---------------------:|----------------------:|
| south        |    102.3 |                41.93 |                  0.05 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 0.05 < 2.58$, гипотеза $H_0$ не отвергается.
:::



**Робастный t-тест (HC3)**

In [47]:
#| output: asis
theta_test(res_hc, 'south', 100, 0.01)



| Переменная   |   Оценка |   Стандартная ошибка (HC3) |   Тестовая статистика |
|:-------------|---------:|---------------------------:|----------------------:|
| south        |    102.3 |                      42.59 |                  0.05 |





::: {.callout-tip title="Вывод" icon="true"}
$\lvert t\rvert = 0.05 < 2.58$, гипотеза $H_0$ не отвергается.
:::

